## 智能体

#### 加载环境

In [ ]:
from dotenv import load_dotenv

# 读取 .env 配置
load_dotenv(override=True)

#### 高级用法

##### 1、系统提示词

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from rich.pretty import pprint

agent = create_agent(
    model='deepseek:deepseek-flash',
    system_prompt="你是耐心的 AI 助手"
)

messages = [HumanMessage("你好")]
resp = agent.invoke({'messages': messages})
# 打印中没有系统提示词
pprint(resp)

##### 2、结构化输出

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel, Field
from rich.pretty import pprint


class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影名")
    year: int = Field(description="发行年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分，满分十分")

model = init_chat_model(
    model='deepseek:deepseek-flash',
    # 需关闭思考模型
    extra_body={
        "thinking": {"type": "disabled"}
    }
)

agent = create_agent(
    model=model,
    # 目前 DS 原生不支持格式化输出
    # response_format=ProviderStrategy(Movie)
    # ToolStrategy 底层是通过工具调用实现
    response_format=ToolStrategy(
        # 和 model.with_structured_output 参数类似
        schema=Movie,
        # 简化 ToolMessage content 的内容，降低 Token 消耗
        tool_message_content="成功提取信息"
    )
)

messages = [HumanMessage("给出电影盗梦空间的信息")]
resp = agent.invoke({'messages': messages})
print(resp["structured_response"])
pprint(resp)

###### 异常处理

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain.agents.structured_output import ToolStrategy, MultipleStructuredOutputsError
from pydantic import BaseModel, Field
from typing import Union
from rich.pretty import pprint


class Person(BaseModel):
    """个人信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")

class EventDetails(BaseModel):
    """活动详情"""
    name: str = Field(description="活动名称")
    date: int = Field(description="活动日期")

def handle_error(error: Exception) -> str:
    print(error)
    if isinstance(error, MultipleStructuredOutputsError):
        return "返回多个结构化相应，请修复"
    raise error

model = init_chat_model(
    model='deepseek:deepseek-flash',
    # 需关闭思考模型
    extra_body={
        "thinking": {"type": "disabled"}
    }
)

agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        schema=Union[Person, EventDetails],
        tool_message_content="成功提取信息",
        # 异常处理，底层核心两条路径
        # 1、反馈给模型，再次生成
        # 2、抛出异常
        # handle_errors=True  # 默认，框架整理提示词给模型纠错
        # handle_errors=False # 抛出异常
        # handle_errors="返回多个结构化相应，请修复" # 自定义提示词给模型纠错
        # handle_errors=(MultipleStructuredOutputsError) # 指定异常执行 True 配置操作，其它抛出异常
        handle_errors=handle_error # 返回字符串，则以返回字符串作为提示词给模型纠错，抛异常的话，则抛出异常
    )
)

messages = [HumanMessage("请提取文本内容：张三 30 岁，将于 2026-07-16 参加公司年会")]
resp = agent.invoke({'messages': messages})
pprint(resp)